# Quantile regression retrieval comparison - corrected revision 1.1

Restart the kernel and Run All after updating. The historical notebook is preserved in `references/pre_methodology_fixes_v1/`. Retrieval methods share corrected code in `src/retrieval_methods.py`; data dimensions and seed 42 are retained.


In [ ]:
import numpy as np
from scipy import sparse

import gurobipy as gp
from gurobipy import GRB
from scipy.sparse.linalg import lsmr, lsqr
import time
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src/result_table_generator.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.retrieval_methods import refine_retrieval

# Same numerical settings; named here for saved run configuration.
INSTANCE_SEED = 42
PROJECTION_SEED = 42
RETRIEVAL_MAX_ITER = 30
RETRIEVAL_TOL = 0.01
AARM_ALPHA = 0.9
AARM_BETA = 0.9


In [ ]:
def generate_quantile_instance(
    m: int,
    p: int,
    density: float = 0.8,
    seed: int | None = None,
):
    """
    Generate one synthetic Quantile Regression instance.

    Parameters
    ----------
    m : int
        Number of observations / records.

    p : int
        Number of explanatory variables (columns of D).

    density : float, default=0.8
        Target proportion of nonzero entries in D.

    seed : int or None, default=None
        Random seed for reproducibility.

    Returns
    -------
    D : ndarray, shape (m, p)
        Design matrix.

    b : ndarray, shape (m,)
        Dependent variable.
    """

    # create a local random number generator
    rng = np.random.default_rng(seed)

    # check input values
    if m <= 0:
        raise ValueError("m must be positive")

    if p <= 0:
        raise ValueError("p must be positive")

    if not 0.0 <= density <= 1.0:
        raise ValueError("density must lie between 0 and 1")

    # initialize D as an all-zero matrix
    D = np.zeros((m, p), dtype=float)

    # calculate the total number of entries
    n_total = m * p

    # calculate the number of nonzero entries
    n_nonzero = int(round(density * n_total))

    # randomly choose the positions of nonzero entries
    nonzero_indices = rng.choice(n_total, size=n_nonzero, replace=False)

    # fill the selected positions with values sampled from U[-1, 1]
    D.flat[nonzero_indices] = rng.uniform(-1.0, 1.0, size=n_nonzero)

    # generate the dependent variable from U[-1, 1]
    b = rng.uniform(-1.0, 1.0, size=m)

    return D, b

In [ ]:
# generate one quantile regression instance
D, b = generate_quantile_instance(m=5000, p=399, density=0.8, seed=INSTANCE_SEED)

# check dimensions
print("D shape:", D.shape)
print("b shape:", b.shape)

# check density
actual_density = np.count_nonzero(D) / D.size

print("number of entries in D:", D.size)
print("number of nonzero entries:", np.count_nonzero(D))
print("actual density:", actual_density)

# check value ranges
print("D min:", D.min())
print("D max:", D.max())
print("b min:", b.min())
print("b max:", b.max())

In [ ]:
def build_quantile_lp(
    D: np.ndarray,
    b: np.ndarray,
    tau: float = 0.2,
):
    """
    Build the LP formulation of a quantile regression instance.

    Parameters
    ----------
    D : ndarray, shape (m, p)
        Design matrix

    b : ndarray, shape (m,)
        Dependent variable

    tau : float, default=0.2
        Quantile level

    Returns
    -------
    A : sparse matrix, shape (m, p + 2m)
        Equality constraint matrix

    c : ndarray, shape (p + 2m,)
        Objective coefficient vector

    bounds : list
        Variable bounds for beta, u_plus, and u_minus
    """

    # get the dimensions of D
    m, p = D.shape

    # check the dimension of b
    if b.shape != (m,):
        raise ValueError("b must have shape (m,)")

    # check the quantile level
    if not 0.0 < tau < 1.0:
        raise ValueError("tau must lie strictly between 0 and 1")

    # convert D to sparse format
    D_sparse = sparse.csr_matrix(D)

    # construct a sparse identity matrix
    I = sparse.eye(m, format="csr")

    # construct A = [D | I | -I]
    A = sparse.hstack((D_sparse, I, -I), format="csr")

    # construct the objective coefficient vector
    c = np.concatenate(
        (np.zeros(p), tau * np.ones(m), (1.0 - tau) * np.ones(m))
    )

    return A, c

In [ ]:
# build the quantile regression LP
A, c = build_quantile_lp(D, b, tau=0.2)

# check dimensions
print("A shape:", A.shape)
print("c shape:", c.shape)

# check the three blocks of the objective
print("beta coefficients:", np.unique(c[: D.shape[1]]))
print(
    "u_plus coefficients:", np.unique(c[D.shape[1] : D.shape[1] + D.shape[0]])
)
print("u_minus coefficients:", np.unique(c[D.shape[1] + D.shape[0] :]))

In [ ]:
def generate_projection_matrix(
    A,
    k: int,
    seed: int | None = None,
):
    """
    Generate a sparse random projection matrix T.

    Parameters
    ----------
    A : sparse matrix, shape (m, n)
        Constraint matrix

    k : int
        Projected dimension

    seed : int or None, default=None
        Random seed for reproducibility

    Returns
    -------
    T : sparse matrix, shape (k, m)
        Sparse random projection matrix
    """

    # create a local random number generator
    rng = np.random.default_rng(seed)

    # get the number of original constraints
    m = A.shape[0]

    # check the projected dimension
    if k <= 0:
        raise ValueError("k must be positive")

    # calculate the density of A
    density_A = A.nnz / (A.shape[0] * A.shape[1])

    # set the density of T
    sigma = density_A / 2.0

    # calculate the total number of entries in T
    n_total = k * m

    # calculate the number of nonzero entries in T
    n_nonzero = int(round(sigma * n_total))

    # randomly choose the positions of nonzero entries
    nonzero_indices = rng.choice(n_total, size=n_nonzero, replace=False)

    # convert the flat indices to row and column indices
    rows = nonzero_indices // m
    cols = nonzero_indices % m

    # calculate the standard deviation of nonzero entries
    std = 1.0 / np.sqrt(k * sigma)

    # sample the nonzero entries from the Gaussian distribution
    values = rng.normal(loc=0.0, scale=std, size=n_nonzero)

    # construct the sparse projection matrix
    T = sparse.csr_matrix((values, (rows, cols)), shape=(k, m))

    return T

In [ ]:
# choose the projection error parameter
epsilon = 0.2

# get the number of original constraints
m = A.shape[0]

# get the total number of decision variables
n_variables = A.shape[1]

# calculate the projected dimension used in the paper
k = int(round((1.0 / epsilon) ** 2) * np.log(float(n_variables)))

# k is not clipped at m; small fixtures may have k >= m.

# generate a sparse random projection matrix
start_time = time.perf_counter()

T = generate_projection_matrix(A, k=k, seed=PROJECTION_SEED)

time_T_generation = time.perf_counter() - start_time

# check the experiment settings
print("epsilon:", epsilon)
print("number of original constraints m:", m)
print("number of decision variables:", n_variables)
print("projected dimension k:", k)
print("projection ratio k/m:", k / m)

# check dimensions
print("T shape:", T.shape)

# check the number of nonzero entries
print("T nnz:", T.nnz)

# check the density of A
density_A = A.nnz / (A.shape[0] * A.shape[1])

print("A density:", density_A)

# check the target density of T
sigma = density_A / 2.0

print("target T density:", sigma)

# check the actual density of T
density_T = T.nnz / (T.shape[0] * T.shape[1])

print("actual T density:", density_T)

# check the projection matrix generation time
print("T generation time:", time_T_generation)

In [ ]:
def project_lp(
    A,
    b: np.ndarray,
    T,
):
    """
    Project the equality constraints of an LP.

    Parameters
    ----------
    A : sparse matrix, shape (m, n)
        Original constraint matrix

    b : ndarray, shape (m,)
        Original right-hand side vector

    T : sparse matrix, shape (k, m)
        Random projection matrix

    Returns
    -------
    A_projected : sparse matrix, shape (k, n)
        Projected constraint matrix

    b_projected : ndarray, shape (k,)
        Projected right-hand side vector
    """

    # check the dimensions of A and b
    if b.shape != (A.shape[0],):
        raise ValueError("b must have shape (m,)")

    # check the dimensions of T and A
    if T.shape[1] != A.shape[0]:
        raise ValueError("T and A have incompatible dimensions")

    # project the constraint matrix
    A_projected = T @ A

    # project the right-hand side vector
    b_projected = T @ b

    return A_projected, b_projected

In [ ]:
def solve_lp(A, b, c, lb, ub, method=-1, model_name="lp_model"):
    """Explicit Gurobi method, one thread; clean up the model after solving."""
    from src.quantile_regression import solve_lp as implementation
    return implementation(A, b, c, lb, ub, method=method, model_name=model_name)


In [ ]:
def build_quantile_bounds(
    m: int,
    p: int,
):
    """
    Build the variable bounds for the quantile regression LP.

    Parameters
    ----------
    m : int
        Number of observations

    p : int
        Number of regression coefficients

    Returns
    -------
    lb : ndarray, shape (p + 2m,)
        Lower bounds

    ub : ndarray, shape (p + 2m,)
        Upper bounds
    """

    # construct the lower bounds
    lb = np.concatenate((np.full(p, -np.inf), np.zeros(m), np.zeros(m)))

    # construct the upper bounds
    ub = np.full(p + 2 * m, GRB.INFINITY)

    return lb, ub

In [ ]:
# project the original equality constraints
start_time = time.perf_counter()
A_projected, b_projected = project_lp(A, b, T)
time_projection = time.perf_counter() - start_time

# get the dimensions of the quantile regression instance
m, p = D.shape

# build the variable bounds
lb, ub = build_quantile_bounds(m=m, p=p)

# solve the projected LP using Gurobi dual simplex
start_time = time.perf_counter()
x_tilde, obj_tilde = solve_lp(A_projected, b_projected, c, lb, ub, method=1)
time_TLP_solve = time.perf_counter() - start_time

In [ ]:
def solve_projection_system(x0, A, b, x_check):
    """Shared corrected implementation; see src.retrieval_methods."""
    from src.retrieval_methods import solve_projection_system as implementation
    return implementation(x0, A, b, x_check)



In [ ]:
def bound_error_sum(x, lb, ub):
    """Shared corrected implementation; see src.retrieval_methods."""
    from src.retrieval_methods import bound_error_sum as implementation
    return implementation(x, lb, ub)



def retrieval_feasibility(x, A, b, lb, ub, tol=0.01, *, eq_tol=None, bound_tol=None):
    """Shared corrected implementation; see src.retrieval_methods."""
    from src.retrieval_methods import retrieval_feasibility as implementation
    return implementation(x, A, b, lb, ub, tol, eq_tol=eq_tol, bound_tol=bound_tol)



def retrieval_start(x_projected, A, b):
    """Shared corrected implementation; see src.retrieval_methods."""
    from src.retrieval_methods import retrieval_start as implementation
    return implementation(x_projected, A, b)



def retrieve_apm(x0, A, b, lb, ub, max_iter=30, delta=0.01, *, trace=None):
    """Shared corrected implementation; see src.retrieval_methods."""
    from src.retrieval_methods import retrieve_apm as implementation
    return implementation(x0, A, b, lb, ub, max_iter, delta, trace=trace)



In [ ]:
def evaluate_retrieval(
    x: np.ndarray,
    A,
    b: np.ndarray,
    c: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
):
    """
    Evaluate a retrieved solution.
    """

    # calculate the objective value
    objective_value = c @ x

    # calculate the equality constraint residual
    equality_residual = np.linalg.norm(A @ x - b)

    # calculate lower-bound violations
    lower_violation = np.maximum(lb - x, 0.0)

    # calculate upper-bound violations
    upper_violation = np.zeros_like(x)

    finite_upper = np.isfinite(ub)

    upper_violation[finite_upper] = np.maximum(
        x[finite_upper] - ub[finite_upper], 0.0
    )

    # calculate the maximum bound violation
    bound_violation = max(np.max(lower_violation), np.max(upper_violation))

    return (objective_value, equality_residual, bound_violation)

In [ ]:
def evaluate_errors(
    x: np.ndarray,
    A,
    b: np.ndarray,
    m: int,
    p: int,
):
    """
    Calculate the average feasibility errors reported in Table 5.

    Parameters
    ----------
    x : ndarray, shape (p + 2m,)
        Retrieved quantile regression solution

    A : sparse matrix, shape (m, p + 2m)
        Original equality constraint matrix

    b : ndarray, shape (m,)
        Original right-hand side vector

    m : int
        Number of observations

    p : int
        Number of regression coefficients

    Returns
    -------
    average_inequality_error : float
        Average nonnegativity violation of u_plus and u_minus

    average_equality_error : float
        Average absolute violation of A x = b
    """

    # check the dimension of the retrieved solution
    if x.shape != (p + 2 * m,):
        raise ValueError(f"x must have shape {(p + 2 * m,)}")

    # check the dimension of b
    if b.shape != (m,):
        raise ValueError(f"b must have shape {(m,)}")

    # extract the positive residual variables
    u_plus = x[p : p + m]

    # extract the negative residual variables
    u_minus = x[p + m : p + 2 * m]

    # calculate violations of u_plus >= 0
    u_plus_violation = np.maximum(-u_plus, 0.0)

    # calculate violations of u_minus >= 0
    u_minus_violation = np.maximum(-u_minus, 0.0)

    # calculate the average inequality error
    average_inequality_error = (
        np.sum(u_plus_violation) + np.sum(u_minus_violation)
    ) / m

    # calculate the original equality residual vector
    equality_error = A @ x - b

    # calculate the average equality error
    average_equality_error = np.mean(np.abs(equality_error))

    return (average_inequality_error, average_equality_error)

In [ ]:
# solve the original LP using Gurobi automatic selection
start_time = time.perf_counter()

x_original, obj_original = solve_lp(
    A, b, c, lb, ub, method=-1, model_name="original_quantile_lp"
)

time_original_solve = time.perf_counter() - start_time

In [ ]:
apm_trace = []
start_time = time.perf_counter()
x_apm, apm_iterations, apm_status = retrieve_apm(
    x0=x_tilde, A=A, b=b, lb=lb, ub=ub, max_iter=RETRIEVAL_MAX_ITER, delta=RETRIEVAL_TOL, trace=apm_trace
)

# Section 3.5 postprocessing is included in retrieval wall time.
x_apm, apm_bisection_iterations, apm_bisection_status = refine_retrieval(
    x_apm, x_tilde, A, b, c, lb, ub, delta=RETRIEVAL_TOL)
time_apm_retrieval = time.perf_counter() - start_time
print("apm bisection:", apm_bisection_status)

(
    apm_objective,
    apm_residual,
    apm_bound_violation,
) = evaluate_retrieval(
    x_apm,
    A,
    b,
    c,
    lb,
    ub,
)

print("APM objective:", apm_objective)
print("APM equality residual:", apm_residual)
print("APM bound violation:", apm_bound_violation)

In [ ]:
if abs(obj_original) > 1e-12:
    relative_objective_error = abs(apm_objective - obj_original) / abs(
        obj_original
    )
    print("Relative objective error:", relative_objective_error)
else:
    relative_objective_error = np.nan
    print("Relative objective error: undefined because obj_original is 0")

In [ ]:
def retrieve_dykstra(x0, A, b, lb, ub, max_iter=30, delta=0.01, *, trace=None):
    """Shared corrected implementation; see src.retrieval_methods."""
    from src.retrieval_methods import retrieve_dykstra as implementation
    return implementation(x0, A, b, lb, ub, max_iter, delta, trace=trace)



In [ ]:
dykstra_trace = []
start_time = time.perf_counter()
x_dykstra, dykstra_iterations, dykstra_status = retrieve_dykstra(
    x0=x_tilde, A=A, b=b, lb=lb, ub=ub, max_iter=RETRIEVAL_MAX_ITER, delta=RETRIEVAL_TOL, trace=dykstra_trace
)

# Section 3.5 postprocessing is included in retrieval wall time.
x_dykstra, dykstra_bisection_iterations, dykstra_bisection_status = refine_retrieval(
    x_dykstra, x_tilde, A, b, c, lb, ub, delta=RETRIEVAL_TOL)
time_dykstra_retrieval = time.perf_counter() - start_time
print("dykstra bisection:", dykstra_bisection_status)

In [ ]:
dykstra_objective, dykstra_residual, dykstra_bound_violation = (
    evaluate_retrieval(x_dykstra, A, b, c, lb, ub)
)

print("Dykstra objective:", dykstra_objective)
print("Dykstra equality residual:", dykstra_residual)
print("Dykstra bound violation:", dykstra_bound_violation)

In [ ]:
print("===== von Neumann APM =====")
print("Iterations:", apm_iterations)
print("Objective:", apm_objective)
print("Equality residual:", apm_residual)
print("Bound violation:", apm_bound_violation)

print("\n===== Dykstra APM =====")
print("Iterations:", dykstra_iterations)
print("Objective:", dykstra_objective)
print("Equality residual:", dykstra_residual)
print("Bound violation:", dykstra_bound_violation)

In [ ]:
def retrieve_aarm(x0, A, b, lb, ub, max_iter=30, delta=0.01, alpha=0.9, beta=0.9, *, trace=None):
    """Shared corrected implementation; see src.retrieval_methods."""
    from src.retrieval_methods import retrieve_aarm as implementation
    return implementation(x0, A, b, lb, ub, max_iter, delta, alpha, beta, trace=trace)



In [ ]:
aarm_trace = []
start_time = time.perf_counter()
x_aarm, aarm_iterations, aarm_status = retrieve_aarm(
    x0=x_tilde,
    A=A,
    b=b,
    lb=lb,
    ub=ub,
    max_iter=RETRIEVAL_MAX_ITER,
    delta=RETRIEVAL_TOL,
    alpha=AARM_ALPHA,
    beta=AARM_BETA,
    trace=aarm_trace,
)

# Section 3.5 postprocessing is included in retrieval wall time.
x_aarm, aarm_bisection_iterations, aarm_bisection_status = refine_retrieval(
    x_aarm, x_tilde, A, b, c, lb, ub, delta=RETRIEVAL_TOL)
time_aarm_retrieval = time.perf_counter() - start_time
print("aarm bisection:", aarm_bisection_status)

print("AARM iterations:", aarm_iterations)
print("AARM stop reason:", aarm_status)

In [ ]:
aarm_objective, aarm_residual, aarm_bound_violation = evaluate_retrieval(
    x_aarm, A, b, c, lb, ub
)

print("AARM objective:", aarm_objective)
print("AARM equality residual:", aarm_residual)
print("AARM bound violation:", aarm_bound_violation)

In [ ]:
# Compare objectives only when ORIGINAL constraints are satisfied.
# The TLP objective is for a different, projected feasible set.
print("===== Original LP =====")
print("Objective:", obj_original)
print("\n===== TLP (projected LP) =====")
print("Objective:", obj_tilde)

for name, x, steps, status, objective, eq_residual, bound_error in [
    (
        "von Neumann APM",
        x_apm,
        apm_iterations,
        apm_status,
        apm_objective,
        apm_residual,
        apm_bound_violation,
    ),
    (
        "Dykstra APM",
        x_dykstra,
        dykstra_iterations,
        dykstra_status,
        dykstra_objective,
        dykstra_residual,
        dykstra_bound_violation,
    ),
    (
        "AARM",
        x_aarm,
        aarm_iterations,
        aarm_status,
        aarm_objective,
        aarm_residual,
        aarm_bound_violation,
    ),
]:
    feasible = retrieval_feasibility(x, A, b, lb, ub, tol=RETRIEVAL_TOL)
    print(f"\n===== {name} =====")
    print("Iterations:", steps)
    print("Stop reason:", status)
    print("Original LP feasible at tolerance:", feasible)
    print(
        "Objective:",
        objective,
        "(comparable)" if feasible else "(infeasible point)",
    )
    print("Equality residual (L2):", eq_residual)
    print("Maximum bound violation:", bound_error)

# Wall-clock times for the same single problem instance.
# Shared data/matrix generation (D, b, A) is excluded from both pipelines.
# This comparison uses wall time; the table generator separately records CPU time.
print("\n===== Wall-clock time (seconds) =====")
print(f"Generate T:           {time_T_generation:.3f}")
print(f"Compute TA and Tb:    {time_projection:.3f}")
print(f"Solve TLP:            {time_TLP_solve:.3f}")
print(f"Solve original LP:    {time_original_solve:.3f}")
print(f"von Neumann retrieval:{time_apm_retrieval:.3f}")
print(f"Dykstra retrieval:    {time_dykstra_retrieval:.3f}")
print(f"AARM retrieval:       {time_aarm_retrieval:.3f}")

# Compare complete RP + retrieval workflows with the direct original solve.
time_projected_base = time_T_generation + time_projection + time_TLP_solve
print("\n===== Complete pipelines (seconds) =====")
print(f"Original LP directly:             {time_original_solve:.3f}")
print(
    f"RP + TLP + von Neumann retrieval: {time_projected_base + time_apm_retrieval:.3f}"
)
print(
    f"RP + TLP + Dykstra retrieval:     {time_projected_base + time_dykstra_retrieval:.3f}"
)
print(
    f"RP + TLP + AARM retrieval:        {time_projected_base + time_aarm_retrieval:.3f} (check feasibility above)"
)

# A ratio above 1 means the Dykstra pipeline ran faster than a direct LP solve.
dykstra_pipeline_time = time_projected_base + time_dykstra_retrieval
print(
    f"Direct / Dykstra pipeline:        {time_original_solve / dykstra_pipeline_time:.3f}x"
)

In [ ]:
# get the dimensions of the quantile regression instance
m, p = D.shape

# calculate the Table 5 errors for von Neumann APM
apm_avgin, apm_avgeq = evaluate_errors(x_apm, A, b, m, p)

# calculate the Table 5 errors for Dykstra APM
dykstra_avgin, dykstra_avgeq = evaluate_errors(x_dykstra, A, b, m, p)

# calculate the Table 5 errors for AARM
aarm_avgin, aarm_avgeq = evaluate_errors(x_aarm, A, b, m, p)

# print the Table 5 feasibility measures
print("===== feasibility measures =====")

print("\nVon Neumann APM")
print("avgin:", apm_avgin)
print("avgeq:", apm_avgeq)

print("\nDykstra APM")
print("avgin:", dykstra_avgin)
print("avgeq:", dykstra_avgeq)

print("\nAARM")
print("avgin:", aarm_avgin)
print("avgeq:", aarm_avgeq)

In [ ]:
dykstra_bound_error_sum = bound_error_sum(
    x_dykstra,
    lb,
    ub,
)

dykstra_equality_error = np.linalg.norm(A @ x_dykstra - b)

dykstra_combined_error = dykstra_bound_error_sum + dykstra_equality_error

print(
    "Dykstra bound-error sum:",
    dykstra_bound_error_sum,
)

print(
    "Dykstra equality error:",
    dykstra_equality_error,
)

print(
    "Dykstra combined error:",
    dykstra_combined_error,
)

print(
    "Required combined error:",
    2.0 * 0.01,
)
print("Both individual tolerances satisfied:", retrieval_feasibility(x_dykstra, A, b, lb, ub, tol=RETRIEVAL_TOL))


The values above belong to the current run. Feasibility requires both the
summed bound violation and equality L2 error to be at most 0.01. Small average
errors or a stagnation status do not establish this condition. AARM implements the
paper's composed reflections; its final iterate can still fail the LP checks.

This notebook retains the original seed-42 data setup (fixed-count predictor
sparsity and a dense response), uses Gurobi and wall timing, and now includes
minimization bisection. It is a single-instance comparison, not the averaged
QR/Diet benchmark. See `docs/methodology-fixes.md` for changes and the preserved
pre-fix notebook under `references/pre_methodology_fixes_v1/`.


Iteration traces below show the actual stopping branch and pre-bisection residuals. Exports run after timing completes. Restart the kernel and Run All to collect a fresh trace.


In [ ]:
# Export only after all timed retrieval/postprocessing calls have completed.
import hashlib
import json
import pandas as pd
from src.result_tables.provenance import environment_metadata
from src.result_tables.reporting import iteration_table
from src.result_tables.retrieval_trace import stopping_explanation

TRACE_OUTPUT_DIR = Path(globals().get('TRACE_OUTPUT_DIR', ROOT / 'results' / 'qr_retrieval_trace'))
TRACE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trace_trials = []
explanations = []
for key, history, x, status, bisection_status, bisection_iterations in [
    ('VN', apm_trace, x_apm, apm_status, apm_bisection_status, apm_bisection_iterations),
    ('Dykstra', dykstra_trace, x_dykstra, dykstra_status, dykstra_bisection_status, dykstra_bisection_iterations),
    ('AARM', aarm_trace, x_aarm, aarm_status, aarm_bisection_status, aarm_bisection_iterations),
]:
    trace_trials.append(dict(problem_type='Quantile Regression', row_size=m,
        epsilon=epsilon, replication=1, formulation_variant='QR',
        instance_seed=INSTANCE_SEED, projection_seed=PROJECTION_SEED, retrieval_trace=history))
    explanations.append(stopping_explanation(history, status,
        final_feasible=retrieval_feasibility(x, A, b, lb, ub, tol=RETRIEVAL_TOL),
        bisection_status=bisection_status, bisection_iterations=bisection_iterations))
comparison_trace = iteration_table(pd.DataFrame(trace_trials))
comparison_trace.to_csv(TRACE_OUTPUT_DIR / 'retrieval_trace.csv', index=False, float_format='%.17g')
trace_configuration = dict(problem_type='Quantile Regression', row_size=m, predictors=p,
    epsilon=epsilon, k=k, instance_seed=INSTANCE_SEED, projection_seed=PROJECTION_SEED,
    max_iterations=RETRIEVAL_MAX_ITER, tolerance=RETRIEVAL_TOL,
    aarm_alpha=AARM_ALPHA, aarm_beta=AARM_BETA, repetitions=1,
    data_generator='reference notebook: fixed-count predictor sparsity; dense response',
    original_solver_method=-1, projected_solver_method=1, solver_threads=1,
    timing='wall time; in-memory trace overhead included; exports excluded',
    environment=environment_metadata(),
    comparison_methods_sha256=hashlib.sha256((ROOT / 'src/retrieval_methods.py').read_bytes()).hexdigest())
(TRACE_OUTPUT_DIR / 'config.json').write_text(json.dumps(trace_configuration, indent=2) + '\n', encoding='utf-8')
trace_log = '\n\n'.join(explanations) + '\n'
(TRACE_OUTPUT_DIR / 'run.log').write_text(trace_log, encoding='utf-8')
print(trace_log)
print('Retrieval trace and configuration saved to:', TRACE_OUTPUT_DIR)
